# Kaggle Olist Brazilian E-Commerce Dataset - Data Integration & Cleaning
This notebook demonstrates:
1. Loading the 4 core datasets (`customers`, `orders`, `order_items`, `order_payments`).
2. Merging them step-by-step into `df_master`.
3. Cleaning `df_master` (filtering delivered orders, converting datetimes, dropping high-null columns, and removing duplicates).
4. Exporting the cleaned master dataset to `Data/Processed/df_master.csv`.

In [1]:
import os
import pandas as pd

# Base directory for raw and processed data
DATA_RAW_DIR = os.path.join("..", "Data", "Raw") if os.path.exists(os.path.join("..", "Data", "Raw")) else os.path.join("Data", "Raw")
DATA_PROCESSED_DIR = os.path.join("..", "Data", "Processed") if os.path.exists(os.path.join("..", "Data", "Processed")) else os.path.join("Data", "Processed")
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)

print(f"Raw Data Directory:       {os.path.abspath(DATA_RAW_DIR)}")
print(f"Processed Data Directory: {os.path.abspath(DATA_PROCESSED_DIR)}")

Raw Data Directory:       c:\Users\evaskant\OneDrive - Netcompany\Desktop\The great Escape\Data\Raw
Processed Data Directory: c:\Users\evaskant\OneDrive - Netcompany\Desktop\The great Escape\Data\Processed


## 1. Load Datasets

In [2]:
print("Loading datasets...")
df_customers = pd.read_csv(os.path.join(DATA_RAW_DIR, "olist_customers_dataset.csv"))
df_orders = pd.read_csv(os.path.join(DATA_RAW_DIR, "olist_orders_dataset.csv"))
df_order_items = pd.read_csv(os.path.join(DATA_RAW_DIR, "olist_order_items_dataset.csv"))
df_order_payments = pd.read_csv(os.path.join(DATA_RAW_DIR, "olist_order_payments_dataset.csv"))

print(f"df_customers shape:      {df_customers.shape}")
print(f"df_orders shape:         {df_orders.shape}")
print(f"df_order_items shape:    {df_order_items.shape}")
print(f"df_order_payments shape: {df_order_payments.shape}")

Loading datasets...
df_customers shape:      (99441, 5)
df_orders shape:         (99441, 8)
df_order_items shape:    (112650, 7)
df_order_payments shape: (103886, 5)


## 2. Step-by-Step Merge

In [3]:
# Step 2.1: Merge Orders with Customers on customer_id
df_step1 = pd.merge(df_orders, df_customers, on="customer_id", how="inner")
print(f"Step 1 (Orders + Customers) shape: {df_step1.shape}")

# Step 2.2: Merge with Order Items on order_id
df_step2 = pd.merge(df_step1, df_order_items, on="order_id", how="inner")
print(f"Step 2 (+ Order Items) shape:      {df_step2.shape}")

# Step 2.3: Merge with Order Payments on order_id -> df_master
df_master = pd.merge(df_step2, df_order_payments, on="order_id", how="inner")
print(f"Final df_master shape (uncleaned): {df_master.shape}")

Step 1 (Orders + Customers) shape: (99441, 12)
Step 2 (+ Order Items) shape:      (112650, 18)
Final df_master shape (uncleaned): (117601, 22)


## 3. Data Cleaning Pipeline
1. Filter for `order_status == 'delivered'`
2. Convert date/timestamp columns to `datetime64[ns]`
3. Drop columns with > 40% missing values
4. Drop duplicate rows

In [4]:
# 1) Filter dataset to keep ONLY 'delivered' orders
df_master = df_master[df_master["order_status"] == "delivered"].copy()

# 2) Convert all date/timestamp columns to pandas datetime
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date"
]
for col in date_cols:
    if col in df_master.columns:
        df_master[col] = pd.to_datetime(df_master[col])

# 3) Drop columns that have more than 40% missing values
missing_threshold = 0.40
high_missing_cols = df_master.columns[df_master.isnull().mean() > missing_threshold]
if len(high_missing_cols) > 0:
    print(f"Dropping columns with >40% missing values: {list(high_missing_cols)}")
    df_master = df_master.drop(columns=high_missing_cols)
else:
    print("No columns exceeded the 40% missing value threshold.")

# 4) Drop any duplicate rows
initial_len = len(df_master)
df_master = df_master.drop_duplicates()
print(f"Dropped {initial_len - len(df_master)} duplicate rows.")

# 5) Print cleaned shape and missing values per column
print(f"\nCleaned df_master shape: {df_master.shape}")
print("\nSum of missing values per column:")
print(df_master.isnull().sum())

No columns exceeded the 40% missing value threshold.
Dropped 0 duplicate rows.

Cleaned df_master shape: (115035, 22)

Sum of missing values per column:
order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                15
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
customer_unique_id                0
customer_zip_code_prefix          0
customer_city                     0
customer_state                    0
order_item_id                     0
product_id                        0
seller_id                         0
shipping_limit_date               0
price                             0
freight_value                     0
payment_sequential                0
payment_type                      0
payment_installments              0
payment_value                     0
dtype: int64


## 4. Export Cleaned Dataset to Processed Folder

In [5]:
output_csv_path = os.path.join(DATA_PROCESSED_DIR, "df_master.csv")
df_master.to_csv(output_csv_path, index=False)
print(f"Successfully saved df_master to: {output_csv_path}")
print(f"File size: {os.path.getsize(output_csv_path) / (1024 * 1024):.2f} MB")

Successfully saved df_master to: ..\Data\Processed\df_master.csv
File size: 37.49 MB
